In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings("ignore")

# Step 1: Identifying frequently used words and phrases in mentor feedback.
- N-gram analayses

In [2]:
# Read the dataset and drop all columns except mentor_feedback_text
train_df = pd.read_csv("data/train.csv")
train_df.head()

,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,...,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,career_success_score,mentor_feedback_text
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,...,62.70,58.84,3,1,24,0,photography,LinkedIn,86.78,Proje kalitesi ve makine öğrenimi konusundaki ...
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,...,42.32,40.54,2,0,46,5,reading,YouTube,46.16,Kodlama ve problem çözme becerileri gelişmekte...
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,...,47.27,82.56,1,2,46,5,cinema,Reddit,84.08,İleri düzey frontend geliştirme becerileri ile...
3,STU_000004,2019,22,2018,Computer Engineering,Tier 1,2.82,54.85,77.80,2,...,78.69,85.05,2,4,49,7,running,Reddit,89.97,Güçlü bir kodlama yeteneği ve backend geliştir...
4,STU_000005,2026,22,2026,Computer Engineering,Tier 3,2.28,72.25,71.97,1,...,27.22,84.29,1,0,119,13,football,X,92.46,Ürün analizi alanına olan tutkusu ve makine öğ...


In [3]:
# 1. Turkish Stopwords (Let's remove unnecessary words: and, or, with, for, etc.)
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

turkish_stopwords = set(stopwords.words("turkish"))
# You can also add extra unnecessary words of your own choosing.
extra_stop_words = ['bir', 'olarak', 'çok', 'da', 'de', 'ise', 'olan', 'bu', 'kadar', 'daha', 'ancak']
all_stop_words = turkish_stopwords.union(extra_stop_words)
feedbacks = train_df["mentor_feedback_text"].dropna().astype(str)


In [4]:
# ---------------------------------------------------------
# STEP 1: EXTRACTING UNIGRAMS (Especially for Adjectives and Nouns)
# ---------------------------------------------------------

print("--- Extracting Unigrams (Adjectives and Nouns) ---")
vectorizer_1 = CountVectorizer(stop_words=list(all_stop_words), max_features=30)
X_1 = vectorizer_1.fit_transform(feedbacks)
words_1 = pd.DataFrame(X_1.sum(axis=0), columns=vectorizer_1.get_feature_names_out()).T
words_1.columns = ["Frequency"]
print(words_1.sort_values(by="Frequency", ascending=False).head(10))
print("\n")


--- Extracting Unigrams (Adjectives and Nouns) ---
             Frequency
fazla             4966
proje             4721
teknik            4044
geliştirme        3547
iletişim          3348
dikkat            3281
veri              3180
güçlü             3131
takım             2972
konusundaki       2739




In [11]:
# ---------------------------------------------------------
# STEP 2: EXTRACTING BIGRAMS (Especially Ability Patterns)
# ---------------------------------------------------------

print("--- MOST COMMONLY USED BINARY WORDS (e.g., machine learning, data structures) ---")
vectorizer_2 = CountVectorizer(stop_words=list(all_stop_words), ngram_range=(2,2), max_features=30)
X_2 = vectorizer_2.fit_transform(feedbacks)
words_2 = pd.DataFrame(X_2.sum(axis=0), columns=vectorizer_2.get_feature_names_out()).T
words_2.columns = ['Frequency']
print(words_2.sort_values(by='Frequency', ascending=False).head(10))
print("\n")

--- MOST COMMONLY USED BINARY WORDS (e.g., machine learning, data structures) ---
                    Frequency
problem çözme            2446
proje kalitesi           1714
makine öğrenimi          1441
dikkat çekici            1396
takım çalışması          1230
yazılım geliştirme       1158
açık kaynak              1114
dikkat çekiyor           1098
veri yapıları             984
veri bilimi               894




In [8]:
# ---------------------------------------------------------
# STEP 3: EXTRACTING TRIGRAMS (Especially Constructive Criticism)
# ---------------------------------------------------------

print("--- MOST COMMONLY USED TRIGRAMS (e.g., should work harder) ---")

vectorizer_3 = CountVectorizer(stop_words=list(all_stop_words), ngram_range=(3,3), max_features=30)
X_3 = vectorizer_3.fit_transform(feedbacks)
words_3 = pd.DataFrame(X_3.sum(axis=0), columns=vectorizer_3.get_feature_names_out()).T
words_3.columns = ['Frequency']
print(words_3.sort_values(by='Frequency', ascending=False).head(10))

--- MOST COMMONLY USED TRIGRAMS (e.g., should work harder) ---
                             Frequency
makine öğrenimi konusundaki        474
iletişim takım çalışması           437
açık kaynak katkıları              356
veri yapıları konusundaki          333
kodlama problem çözme              323
veri bilimi alanında               323
yazılım geliştirme alanında        321
problem çözme becerileri           312
proje kalitesi üzerinde            303
veri bilimi alanındaki             283


In [13]:
words_1.to_csv('words_1.csv')
words_2.to_csv('words_2.csv')
words_3.to_csv('words_3.csv')

# Step 2: NLP Baseline

In [21]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

In [27]:
class NLPFeatureEngineer(BaseEstimator, TransformerMixin):
    """
        A custom converter class that extracts Natural Language Processing (NLP) based features using only the 'mentor_feedback_text' column.
    """
    def __init__(self):
        # 1. Ability Patterns
        self.tech_skills = [
            'makine öğrenimi', 'veri yapıları', 'veri bilimi', 'veri analizi', 
            'backend', 'frontend', 'bulut', 'yazılım', 'açık kaynak', 'github', 
            'sql', 'yapay zeka', 'arka uç',
            # Copied from the columns (scores) of the dataset.
            'devops', 'cloud', 'kodlama', 'problem çözme', 'algoritma',
            # Copied from target role column 
            'mobil', 'siber güvenlik', 'data', 'analist', 'mühendis', 'developer'
        ]
        self.soft_skills = [
            'iletişim', 'takım çalışması', 'problem çözme', 'mülakat', 'proje', 'takım',
            # Added from columns of the dataset
            'liderlik', 'sunum', 'ekip', 'zaman yönetimi', 'analitik düşünme', 'yönetim', 'ikna'
        ]

        # 2. Signal Patterns
        self.positive_signals = [
            'etkileyici', 'güçlü', 'dikkat çekici', 'dikkat çekiyor', 
            'dikkat çeken', 'sağlam temel', 'aday haline', 'yüksek',
            # Added 
            'başarılı', 'harika', 'potansiyeli', 'iyi seviyede', 'gelişimi çok iyi', 
            'mükemmel', 'tatmin edici', 'beklentileri aşıyor', 'yetkin'
        ]
        self.improvement_signals = [
            'faydalı olacaktır', 'fazla pratik', 'deneyim kazanması', 
            'pratik yapması', 'üzerinde fazla',
            # Most commen words
            'geliştirmeli', 'eksik', 'odaklanmalı', 'çalışmalı', 'yetersiz', 
            'zayıf', 'beklentilerin altında', 'artırmalı', 'ihtiyacı var'
        ]

        # 3. Contextual Patterns
        self.contrast_words = [
            'ancak', 'fakat', 'rağmen', 'gösterse de',
            'bununla birlikte', 'yine de', 'yalnız'
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        # Convert text to lowercase and fill in blanks with 'no_feedback'.
        feedback_lower = df['mentor_feedback_text'].fillna('no_feedback').astype(str).str.lower()

        # --- CALCULATION LOGIC ---
        # Increase the score if both the word 'ability' and the word 'signal' appear in the text.

        # Technical Praise and Criticism
        df['tech_praise_score'] = feedback_lower.apply(
            lambda text: sum(1for w in self.tech_skills if w in text and any(p in text for p in self.positive_signals))
        )
        df['tech_criticism_score'] = feedback_lower.apply(
            lambda text: sum(1 for w in self.tech_skills if w in text and any(n in text for n in self.improvement_signals))
        )

        # Social Skills: Praise and Criticism
        df['soft_praise_score'] = feedback_lower.apply(
            lambda text: sum(1 for w in self.soft_skills if w in text and any(p in text for p in self.positive_signals))
        )
        df['soft_criticism_score'] = feedback_lower.apply(
            lambda text: sum(1 for w in self.soft_skills if w in text and any(n in text for n in self.improvement_signals))
        )

        # Capturing the contrast (the beginning of the criticism)
        df['has_contrast'] = feedback_lower.apply(
            lambda text: 1 if any(c in text for c in self.contrast_words) else 0
        )

        # Basic Text Length Metrics
        df['feedback_word_count'] = feedback_lower.str.split().str.len()
        
        return df


In [28]:
# NLP sınıfını oluştur ve sadece train_df üzerinde test et
nlp_engineer = NLPFeatureEngineer()
df_nlp_eklenmis = nlp_engineer.transform(train_df)

# Çıkan yeni NLP skorlarını ve metni yan yana gör
print(df_nlp_eklenmis[['mentor_feedback_text', 'tech_praise_score', 'tech_criticism_score', 'has_contrast']].head())

                                         mentor_feedback_text  \
student_id                                                      
STU_000001  Proje kalitesi ve makine öğrenimi konusundaki ...   
STU_000002  Kodlama ve problem çözme becerileri gelişmekte...   
STU_000003  İleri düzey frontend geliştirme becerileri ile...   
STU_000004  Güçlü bir kodlama yeteneği ve backend geliştir...   
STU_000005  Ürün analizi alanına olan tutkusu ve makine öğ...   

            tech_praise_score  tech_criticism_score  has_contrast  
student_id                                                         
STU_000001                  4                     4             1  
STU_000002                  0                     3             1  
STU_000003                  1                     0             1  
STU_000004                  2                     0             1  
STU_000005                  1                     0             1  


In [ ]:
df_nlp_eklenmis